In [ ]:
# set up the environment
# python3 -m venv .venv
!pip install 'kagglehub[pandas-datasets]'

import kagglehub
import shutil
import os

In [ ]:
# Download dataset
cache_path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
target_path = "../data/the-movies-dataset"
os.makedirs(target_path, exist_ok=True)

# Copy dataset from cache to your project folder
shutil.copytree(cache_path, target_path, dirs_exist_ok=True)
print("Dataset copied to:", target_path)

In [ ]:
import pandas as pd
import os

os.chdir('/Users/saurav/Documents/Spring-2026/CS_4641_Machine_Learning/movie-recommendation-system/data')
print(os.getcwd())


movies = pd.read_csv("./movies_metadata.csv", low_memory=False)
keywords = pd.read_csv("./keywords.csv")
credits = pd.read_csv("./credits.csv")
links = pd.read_csv("./links.csv")
links_small = pd.read_csv("./links_small.csv")
ratings = pd.read_csv("./ratings_small.csv")


movies.info()
movies.head()


In [ ]:
# Convert to numeric, set errors='coerce' to turn invalid values into NaN
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
ratings['movieId'] = pd.to_numeric(ratings['movieId'], errors='coerce')

# Drop rows with NaN in 'id' or 'movieId'
movies = movies.dropna(subset=['id']).copy()
ratings = ratings.dropna(subset=['movieId']).copy()

# Now safely convert to int64
movies['id'] = movies['id'].astype('int64')
ratings['movieId'] = ratings['movieId'].astype('int64')

print(movies['id'].dtype)
print(ratings['movieId'].dtype)
print(ratings['movieId'].head())
# Remove rows where conversion produced NaN before int cast
movies = movies.dropna(subset=['id']).copy()
ratings = ratings.dropna(subset=['movieId']).copy()

# Convert both columns to standard int64 (non-nullable)
movies['id'] = movies['id'].astype('int64')
ratings['movieId'] = ratings['movieId'].astype('int64')

print(movies['id'].dtype)
print(ratings['movieId'].dtype)
print(ratings['movieId'].head())
# Remove duplicate/corrupted entries (based on 'id')
movies = movies.drop_duplicates(subset='id')

# Remove ratings with missing/null values
ratings = ratings.dropna(subset=['rating'])

# Remove users with very few ratings (less than 3)
user_rating_counts = ratings['userId'].value_counts()
active_users = user_rating_counts[user_rating_counts >= 3].index
ratings = ratings[ratings['userId'].isin(active_users)]

# Print cleaned data shapes
print('Movies shape:', movies.shape)
print('Ratings shape:', ratings.shape)

ratings.head()

ratings['movieId'].nunique() # there are 9066 unique movies in the ratings dataset, which is a subset of the movies in the movies_metadata dataset

# Optimize cleaning: remove movies not in ratings and vice versa
movie_ids_in_ratings = set(ratings['movieId'].unique())
movie_ids_in_movies = set(movies['id'].unique())

# Movies in movies_metadata but not in ratings
to_remove_from_movies = movie_ids_in_movies - movie_ids_in_ratings
if to_remove_from_movies:
    movies = movies[~movies['id'].isin(to_remove_from_movies)]

# Print cleaned data shapes after sync
print('Movies shape after sync:', movies.shape)
print('Ratings shape after sync:', ratings.shape)

In [ ]:
import matplotlib.pyplot as plt

# 1. Correlation between movie popularity and average rating (scatter plot)
ratings_summary = ratings.groupby('movieId').agg({'rating': ['mean', 'count']})
ratings_summary.columns = ['avg_rating', 'rating_count']
plt.figure(figsize=(8,6))
plt.scatter(ratings_summary['rating_count'], ratings_summary['avg_rating'], alpha=0.5)
plt.title('Movie Popularity vs. Average Rating')
plt.xlabel('Number of Ratings')
plt.ylabel('Average Rating')
plt.show()

# 2. Number of ratings per user (histogram)
user_rating_counts = ratings['userId'].value_counts()
plt.figure(figsize=(8,6))
plt.hist(user_rating_counts, bins=30)
plt.title('Number of Ratings per User')
plt.xlabel('Ratings per User')
plt.ylabel('Number of Users')
plt.show()

# 3. Distribution of movie ratings (histogram)
plt.figure(figsize=(8,6))
plt.hist(ratings['rating'], bins=30)
plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# User-based collaborative filtering using cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Create user-item rating matrix
user_item_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')

# Fill NaN with 0 (for cosine similarity)
user_item_matrix_filled = user_item_matrix.fillna(0)

# Choose a target user (example: first user in the dataset)
target_user_id = user_item_matrix_filled.index[0]
target_user_vector = user_item_matrix_filled.loc[target_user_id].values.reshape(1, -1)

# Compute cosine similarity between target user and all other users
similarities = cosine_similarity(target_user_vector, user_item_matrix_filled.values)[0]

# Get top N similar users (excluding the target user)
top_n = 5
similar_users_idx = np.argsort(similarities)[::-1][1:top_n+1]
similar_users = user_item_matrix_filled.index[similar_users_idx]

print(f"Top {top_n} users similar to user {target_user_id}:", similar_users.tolist())

# Aggregate ratings from similar users for movies the target user hasn't rated
unrated_movies = user_item_matrix_filled.columns[user_item_matrix_filled.loc[target_user_id] == 0]
recommendation_scores = user_item_matrix_filled.loc[similar_users, unrated_movies].mean(axis=0)

# Recommend top movies
top_recommendations = recommendation_scores.sort_values(ascending=False).head(10) # type: ignore
print("Top recommended movies for user", target_user_id)
print(top_recommendations)

# Optionally, show movie titles
recommended_movie_ids = top_recommendations.index
recommended_titles = movies[movies['id'].isin(recommended_movie_ids)][['id', 'title']]
print("Recommended movie titles:")
print(recommended_titles)

In [ ]:
# Presentation of recommendations for user a with genre and rating
# 1. User a's top rated movies (with genre and rating)
user_a_id = target_user_id
user_a_ratings = ratings[ratings['userId'] == user_a_id]
top_rated_a = user_a_ratings.sort_values(by='rating', ascending=False).head(10)
top_rated_a_info = movies[movies['id'].isin(top_rated_a['movieId'])][['id', 'title', 'genres']]
top_rated_a_info = top_rated_a_info.merge(top_rated_a[['movieId', 'rating']], left_on='id', right_on='movieId')
print(f"User {user_a_id}'s top rated movies:")
print(top_rated_a_info[['title', 'genres', 'rating']])

# 2. Most similar user (user b) and their top rated movies (with genre and rating)
user_b_id = similar_users[0]
user_b_ratings = ratings[ratings['userId'] == user_b_id]
top_rated_b = user_b_ratings.sort_values(by='rating', ascending=False).head(10)
top_rated_b_info = movies[movies['id'].isin(top_rated_b['movieId'])][['id', 'title', 'genres']]
top_rated_b_info = top_rated_b_info.merge(top_rated_b[['movieId', 'rating']], left_on='id', right_on='movieId')
print(f"Most similar user to user {user_a_id} is user {user_b_id}")
print(f"User {user_b_id}'s top rated movies:")
print(top_rated_b_info[['title', 'genres', 'rating']])

# 3. Recommendations for user a (show names, genre, and predicted rating)
recommended_titles = movies[movies['id'].isin(top_recommendations.index)][['id', 'title', 'genres']]
recommended_titles = recommended_titles.merge(top_recommendations.rename('predicted_rating').reset_index(), left_on='id', right_on='movieId')
print(f"Top recommended movies for user {user_a_id}:")
print(recommended_titles[['title', 'genres', 'predicted_rating']])

In [ ]:
# Model evaluation: Predict which movie out of 10 a user has watched
import random

# Select a validation user (not the target user)
validation_user_id = random.choice([uid for uid in user_item_matrix_filled.index if uid != target_user_id])
validation_user_rated = ratings[ratings['userId'] == validation_user_id]['movieId'].unique()

# Pick 1 movie the user has watched
watched_movie = random.choice(validation_user_rated)

# Pick 9 movies the user has NOT watched
all_movie_ids = set(user_item_matrix_filled.columns)
not_watched_movies = list(all_movie_ids - set(validation_user_rated))
not_watched_sample = random.sample(not_watched_movies, 9)

# Combine for candidate set
candidate_movies = [watched_movie] + not_watched_sample
random.shuffle(candidate_movies)

# Predict scores for validation user
validation_vector = user_item_matrix_filled.loc[validation_user_id].values.reshape(1, -1)
similarities_val = cosine_similarity(validation_vector, user_item_matrix_filled.values)[0]
top_similar_users_val_idx = np.argsort(similarities_val)[::-1][1:top_n+1]
top_similar_users_val = user_item_matrix_filled.index[top_similar_users_val_idx]

# Aggregate scores for candidate movies
candidate_scores = user_item_matrix_filled.loc[top_similar_users_val, candidate_movies].mean(axis=0)

# Find the movie with the highest predicted score
predicted_movie = candidate_movies[np.argmax(candidate_scores)]

# Show results
candidate_titles = movies[movies['id'].isin(candidate_movies)][['id', 'title']]
print(f"Validation user: {validation_user_id}")
print(f"Candidate movies: {candidate_titles}")
print(f"Actual watched movie: {watched_movie}")
print(f"Model's predicted movie: {predicted_movie}")
if predicted_movie == watched_movie:
    print("Model prediction: CORRECT")
else:
    print("Model prediction: INCORRECT")

In [ ]:
# Model evaluation: Plot accuracy across 50 validation users
import matplotlib.pyplot as plt

validation_users = [uid for uid in user_item_matrix_filled.index if uid != target_user_id]
random.shuffle(validation_users)
validation_users_sample = validation_users[:50]

accuracies = []
for validation_user_id in validation_users_sample:
    validation_user_rated = ratings[ratings['userId'] == validation_user_id]['movieId'].unique()
    if len(validation_user_rated) == 0:
        accuracies.append(np.nan)
        continue
    watched_movie = random.choice(validation_user_rated)
    all_movie_ids = set(user_item_matrix_filled.columns)
    not_watched_movies = list(all_movie_ids - set(validation_user_rated))
    if len(not_watched_movies) < 9:
        accuracies.append(np.nan)
        continue
    not_watched_sample = random.sample(not_watched_movies, 9)
    candidate_movies = [watched_movie] + not_watched_sample
    random.shuffle(candidate_movies)
    validation_vector = user_item_matrix_filled.loc[validation_user_id].values.reshape(1, -1)
    similarities_val = cosine_similarity(validation_vector, user_item_matrix_filled.values)[0]
    top_similar_users_val_idx = np.argsort(similarities_val)[::-1][1:top_n+1]
    top_similar_users_val = user_item_matrix_filled.index[top_similar_users_val_idx]
    candidate_scores = user_item_matrix_filled.loc[top_similar_users_val, candidate_movies].mean(axis=0)
    predicted_movie = candidate_movies[np.argmax(candidate_scores)]
    accuracies.append(int(predicted_movie == watched_movie))

# Remove NaN values
accuracies_clean = [acc for acc in accuracies if not np.isnan(acc)]

# Plot results
plt.figure(figsize=(8,6))
plt.plot(range(1, len(accuracies_clean)+1), accuracies_clean, marker='o')
plt.title('Model Validation Accuracy Across 50 Users')
plt.xlabel('Validation User Index')
plt.ylabel('Correct Prediction (1=Yes, 0=No)')
plt.ylim(-0.1, 1.1)
plt.grid(True)
plt.show()

# Print overall accuracy
overall_accuracy = np.mean(accuracies_clean)
print(f"Overall accuracy across 50 users: {overall_accuracy:.2f}")

In [ ]:
# Enhanced model: Item-item similarity using genres and keywords, then gradient boosting with user-user similarity
import ast
import random
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

# Prepare item features (genres + keywords)
def parse_genres(genres_str):
    try:
        genres_list = [d['name'] for d in ast.literal_eval(genres_str)]
        return ' '.join(genres_list)
    except:
        return ''

movies['genres_str'] = movies['genres'].apply(parse_genres)
keywords_dict = dict(zip(keywords['id'], keywords['keywords']))

def parse_keywords(movie_id):
    try:
        kw_list = [d['name'] for d in ast.literal_eval(keywords_dict.get(movie_id, '[]'))]
        return ' '.join(kw_list)
    except:
        return ''

movies['keywords_str'] = movies['id'].apply(parse_keywords)
movies['item_features'] = movies['genres_str'] + ' ' + movies['keywords_str']

# Vectorize item features
vectorizer = CountVectorizer(stop_words='english')
item_feature_matrix = vectorizer.fit_transform(movies['item_features'])

# Compute item-item cosine similarity
item_cosine_sim = cos_sim(item_feature_matrix)

# Map movieId to index in movies
movie_id_to_idx = dict(zip(movies['id'], range(len(movies))))

# Gradient Boosted Model evaluation across 50 users
top_n = 10  # Define top_n if not already set elsewhere

validation_users = [uid for uid in user_item_matrix_filled.index if uid != target_user_id]
random.shuffle(validation_users)
validation_users_sample = validation_users[:50]

X = []
y = []

for validation_user_id in validation_users_sample:
    validation_user_rated = ratings[ratings['userId'] == validation_user_id]['movieId'].unique()
    if len(validation_user_rated) == 0:
        continue

    watched_movie = random.choice(validation_user_rated)
    all_movie_ids = set(user_item_matrix_filled.columns)
    not_watched_movies = list(all_movie_ids - set(validation_user_rated))
    if len(not_watched_movies) < 9:
        continue

    not_watched_sample = random.sample(not_watched_movies, 9)
    candidate_movies = [watched_movie] + not_watched_sample
    random.shuffle(candidate_movies)

    validation_vector = user_item_matrix_filled.loc[validation_user_id].values.reshape(1, -1)
    similarities_val = cos_sim(validation_vector, user_item_matrix_filled.values)[0]
    top_similar_users_val_idx = np.argsort(similarities_val)[::-1][1:top_n + 1]
    top_similar_users_val = user_item_matrix_filled.index[top_similar_users_val_idx]

    user_scores_series = user_item_matrix_filled.loc[top_similar_users_val, candidate_movies].mean(axis=0)

    item_scores = []
    for m in candidate_movies:
        idx = movie_id_to_idx.get(m, None)
        if idx is None:
            item_scores.append(0)
        else:
            watched_idxs = [
                movie_id_to_idx[mid]
                for mid in validation_user_rated
                if mid in movie_id_to_idx
            ]
            sim = item_cosine_sim[idx, watched_idxs].mean() if watched_idxs else 0
            item_scores.append(sim)

    for i, m in enumerate(candidate_movies):
        # BUG FIX 2: use movie id m to index the Series, not integer i
        X.append([user_scores_series[m], item_scores[i]])
        y.append(int(m == watched_movie))

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Gradient Boosting Classifier
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)

# Evaluate accuracy on held-out test set
preds = gb.predict(X_test)
accuracy = np.mean(preds == y_test)
print(f"Gradient Boosted Model accuracy across 50 users: {accuracy:.2f}")